In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"          # same value you used in notebook 01
LAND = f"/Volumes/{CATALOG}/bronze/landing"

def save(df, schema, name):
    full = f"{CATALOG}.{schema}.{name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full))
    print(f"{full}: {spark.table(full).count()} rows")

In [0]:
w = spark.read.option("multiLine", True).json(f"{LAND}/weather_*.json")
w.printSchema()

root
 |-- city: string (nullable = true)
 |-- daily: struct (nullable = true)
 |    |-- precipitation_sum: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_max: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_min: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- daily_units: struct (nullable = true)
 |    |-- precipitation_sum: string (nullable = true)
 |    |-- temperature_2m_max: string (nullable = true)
 |    |-- temperature_2m_min: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = tru

In [0]:
weather = (w
    .select("city", "daily", F.posexplode("daily.time").alias("i", "weather_date"))
    .select(
        "city",
        F.col("weather_date").cast("date").alias("weather_date"),
        F.expr("daily.temperature_2m_max[i]").cast("double").alias("temp_max_c"),
        F.expr("daily.temperature_2m_min[i]").cast("double").alias("temp_min_c"),
        F.expr("daily.precipitation_sum[i]").cast("double").alias("precipitation_mm"))
    .dropDuplicates(["city", "weather_date"]))

save(weather, "silver", "weather")
display(spark.table(f"{CATALOG}.silver.weather").orderBy("city", "weather_date").limit(10))

workspace.silver.weather: 155 rows


city,weather_date,temp_max_c,temp_min_c,precipitation_mm
Bengaluru,2026-09-23,27.1,20.1,1.3
Bengaluru,2026-09-24,29.0,20.0,1.6
Bengaluru,2026-09-25,27.3,20.8,6.7
Bengaluru,2026-09-26,28.2,19.7,0.2
Bengaluru,2026-09-27,29.6,19.4,0.1
Bengaluru,2026-09-28,28.6,21.1,3.3
Bengaluru,2026-09-29,28.2,19.7,13.5
Bengaluru,2026-09-30,28.3,19.5,14.3
Bengaluru,2026-10-01,28.5,20.5,1.2
Bengaluru,2026-10-02,28.8,20.3,0.0


In [0]:
silver_orders = spark.table(f"{CATALOG}.silver.orders")
silver_products = spark.table(f"{CATALOG}.silver.products")

# dim_date: one row per day between the first and last order
rng = silver_orders.agg(F.min("order_date").alias("lo"), F.max("order_date").alias("hi")).first()
dim_date = (spark.sql(
        f"SELECT explode(sequence(DATE'{rng.lo}', DATE'{rng.hi}', INTERVAL 1 DAY)) AS full_date")
    .select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.year("full_date").alias("year"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofmonth("full_date").alias("day"),
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.dayofweek("full_date").isin(1, 7).alias("is_weekend")))

# dim_city: key is a hash of the name, so it stays the same on every run
dim_city = (silver_orders.select("city").distinct()
            .withColumn("city_key", F.abs(F.hash("city")))
            .select("city_key", "city"))

dim_product = silver_products.select(
    F.col("product_id").alias("product_key"), "product_name", "category", "unit_price")

# dim_customer: latest name and first order date (we upgrade this to SCD Type 2 in step 10c)
dim_customer = (silver_orders.groupBy("customer_id")
    .agg(F.expr("max_by(customer_name, order_date)").alias("customer_name"),
         F.min("order_date").alias("first_order_date"))
    .withColumnRenamed("customer_id", "customer_key"))

save(dim_date, "gold", "dim_date")
save(dim_city, "gold", "dim_city")
save(dim_product, "gold", "dim_product")
save(dim_customer, "gold", "dim_customer")

workspace.gold.dim_date: 31 rows
workspace.gold.dim_city: 6 rows
workspace.gold.dim_product: 20 rows
workspace.gold.dim_customer: 501 rows


In [0]:
fact = (silver_orders
    .join(dim_city, "city")
    .join(dim_product.select("product_key", "unit_price"),
          F.col("product_id") == F.col("product_key"))
    .select(
        "order_id",
        F.date_format("order_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("customer_id").alias("customer_key"),
        "product_key",
        "city_key",
        "payment_type",
        "quantity",
        "unit_price",
        (F.col("quantity") * F.col("unit_price")).cast("decimal(12,2)").alias("total_amount")))
save(fact, "gold", "fact_orders")

fact_t = spark.table(f"{CATALOG}.gold.fact_orders")
sales = (fact_t
    .join(dim_date.select("date_key", "full_date"), "date_key")
    .join(dim_city, "city_key")
    .groupBy("full_date", "city")
    .agg(F.count("*").alias("orders"),
         F.sum("quantity").alias("units"),
         F.sum("total_amount").alias("revenue"),
         F.round(F.avg("total_amount"), 2).alias("avg_order_value")))

wx = spark.table(f"{CATALOG}.silver.weather").withColumnRenamed("city", "w_city")
summary = (sales.join(wx, (sales.city == wx.w_city) & (sales.full_date == wx.weather_date), "left")
    .select(F.col("full_date").alias("order_date"), "city", "orders", "units", "revenue",
            "avg_order_value", "temp_max_c", "temp_min_c", "precipitation_mm",
            (F.coalesce("precipitation_mm", F.lit(0.0)) > 1).alias("is_rainy")))
save(summary, "gold", "daily_sales_summary")

workspace.gold.fact_orders: 15500 rows
workspace.gold.daily_sales_summary: 186 rows


In [0]:
fact = (silver_orders
    .join(dim_city, "city")
    .join(dim_product.select("product_key", "unit_price"),
          F.col("product_id") == F.col("product_key"))
    .select(
        "order_id",
        F.date_format("order_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("customer_id").alias("customer_key"),
        "product_key",
        "city_key",
        "payment_type",
        "quantity",
        "unit_price",
        (F.col("quantity") * F.col("unit_price")).cast("decimal(12,2)").alias("total_amount")))
save(fact, "gold", "fact_orders")

fact_t = spark.table(f"{CATALOG}.gold.fact_orders")
sales = (fact_t
    .join(dim_date.select("date_key", "full_date"), "date_key")
    .join(dim_city, "city_key")
    .groupBy("full_date", "city")
    .agg(F.count("*").alias("orders"),
         F.sum("quantity").alias("units"),
         F.sum("total_amount").alias("revenue"),
         F.round(F.avg("total_amount"), 2).alias("avg_order_value")))

wx = spark.table(f"{CATALOG}.silver.weather").withColumnRenamed("city", "w_city")
summary = (sales.join(wx, (sales.city == wx.w_city) & (sales.full_date == wx.weather_date), "left")
    .select(F.col("full_date").alias("order_date"), "city", "orders", "units", "revenue",
            "avg_order_value", "temp_max_c", "temp_min_c", "precipitation_mm",
            (F.coalesce("precipitation_mm", F.lit(0.0)) > 1).alias("is_rainy")))
save(summary, "gold", "daily_sales_summary")

workspace.gold.fact_orders: 15500 rows
workspace.gold.daily_sales_summary: 186 rows


In [0]:
f = spark.table(f"{CATALOG}.gold.fact_orders")
s = spark.table(f"{CATALOG}.silver.orders")
ds = spark.table(f"{CATALOG}.gold.daily_sales_summary")

fact_rev = float(f.agg(F.sum("total_amount")).first()[0])
sum_rev = float(ds.agg(F.sum("revenue")).first()[0])

checks = {
    "fact rows equal silver rows (no join loss)": abs(f.count() - s.count()),
    "no null keys in fact": f.filter("date_key IS NULL OR customer_key IS NULL OR product_key IS NULL OR city_key IS NULL").count(),
    "total_amount is positive": f.filter("total_amount IS NULL OR total_amount <= 0").count(),
    "order_id unique in fact": f.count() - f.select("order_id").distinct().count(),
    "summary revenue equals fact revenue": 0 if abs(fact_rev - sum_rev) < 0.01 else 1,
    "city_key unique": spark.table(f"{CATALOG}.gold.dim_city").count()
                       - spark.table(f"{CATALOG}.gold.dim_city").select("city_key").distinct().count(),
}
for name, bad in checks.items():
    print(("PASS " if bad == 0 else "FAIL ") + name)
assert all(v == 0 for v in checks.values()), "Gold quality checks failed"

PASS fact rows equal silver rows (no join loss)
PASS no null keys in fact
PASS total_amount is positive
PASS order_id unique in fact
PASS summary revenue equals fact revenue
PASS city_key unique


In [0]:
display(spark.sql(f"""
  SELECT count(*) AS orders, sum(total_amount) AS revenue
  FROM {CATALOG}.gold.fact_orders"""))

orders,revenue
15500,109441103.50


In [0]:
silver_orders_t = spark.table(f"{CATALOG}.silver.orders")

# one existing order with a corrected quantity...
changed = (silver_orders_t.limit(1)
           .withColumn("quantity", F.when(F.col("quantity") == 5, 4).otherwise(5)))

# ...and one brand-new order
new = spark.sql("""SELECT 'MERGE-0001' AS order_id, DATE'2026-10-22' AS order_date,
                          1001 AS customer_id, 'Merge Test' AS customer_name, 'Mumbai' AS city,
                          1 AS product_id, 2 AS quantity, 'UPI' AS payment_type""")

updates = changed.unionByName(new)
updates.createOrReplaceTempView("order_updates")
display(updates)

version_before = (spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.orders")
                  .agg(F.max("version")).first()[0])
rows_before = silver_orders_t.count()
print("version before:", version_before, "| rows before:", rows_before)

order_id,order_date,customer_id,customer_name,city,product_id,quantity,payment_type
20260923-00000,2026-09-23,1414,Aryan Maharaj,Mumbai,6,5,COD
MERGE-0001,2026-10-22,1001,Merge Test,Mumbai,1,2,UPI


version before: 1 | rows before: 15500


In [0]:
spark.sql(f"""
  MERGE INTO {CATALOG}.silver.orders AS t
  USING order_updates AS s
  ON t.order_id = s.order_id
  WHEN MATCHED AND t.quantity <> s.quantity THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")

print("rows after:", spark.table(f"{CATALOG}.silver.orders").count())
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.orders")
        .select("version", "operation", "operationMetrics").limit(3))

rows after: 15501


version,operation,operationMetrics
2,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 6242, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6102, materializeSourceTimeMs -> 833, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2237, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2909)"
1,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 136226, numDeletionVectorsRemoved -> 0, numOutputRows -> 15500, numOutputBytes -> 136226)"
0,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 15500, numOutputBytes -> 136226)"


In [0]:
spark.sql(f"RESTORE TABLE {CATALOG}.silver.orders TO VERSION AS OF {version_before}")
print("restored. rows:", spark.table(f"{CATALOG}.silver.orders").count())

restored. rows: 15500
